<h1 style="background-color: #2d90b467; padding: 10px; border-radius: 5px; font-weight: bold; color: #efeacf; text-align: center;">
    Project
</h1>

# 🏠 House Price Prediction

## Introduction

This project focuses on analyzing House Prices, their features and developing a machine learning model to predict the price of the Houses based on their attributes(area, bedrooms, bathrooms etc).

The project begins with exploratory data analysis to understand the dataset, identify patterns in House Prices, and examine relationships between Prices and various features. Various statistical analysis and visualizations are used to gain insights into House Prices and trends.

After exploring the data, a Regression model is developed using Linear Regression to predict the House Pricces based on its features. The model is evaluated using appropriate regression metrics to understand how well it performs on unseen data.

The project also demonstrates how data analysis and machine learning can be combined to turn House Price data into useful insights and predictions.

### 🎯 Objectives

* Analyze house prices across different cities and property types.
* Examine the relationship between area (sqft) and price.
* Identify patterns and trends in the dataset.
* Perform data preprocessing and feature engineering.
* Build a machine learning model to predict house prices.
* Evaluate the performance of the regression model.
* Use the trained model to make predictions for new house data.

---

### 📚 Import Libraries

In [121]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---

### 🔃 Load Dataset

In [122]:
with open("data/data.dat", "r") as f:
    content = f.read()

# The file uses literal NaN instead of null, which isn't valid JSON
content = content.replace("NaN", "null")

data = json.loads(content)
df = pd.DataFrame(data["houses"])

---

### 🔍 Data Exploration and Preparation

#### - Check shape


In [123]:
df.info()
print("\nrows, columns :",df.shape)
missing = df.isnull().sum()
print("\nNull values :\n",missing[missing>0])


<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area          4601 non-null   object 
 1   yr_renovated  229 non-null    float64
 2   price         4601 non-null   float64
 3   waterfront    4601 non-null   int64  
 4   floors        4601 non-null   float64
 5   rooms         4601 non-null   str    
 6   address       4601 non-null   str    
 7   date          4601 non-null   str    
 8   yr_built      4601 non-null   int64  
 9   condition     4601 non-null   int64  
 10  view          4601 non-null   int64  
dtypes: float64(3), int64(4), object(1), str(3)
memory usage: 395.5+ KB

rows, columns : (4601, 11)

Null values :
 yr_renovated    4372
dtype: int64


#### - First 5 Rows

In [124]:
df.head()

,area,yr_renovated,price,waterfront,floors,rooms,address,date,yr_built,condition,view
0,"{'sqft_basement': 0, 'sqft_above': 1340, 'sqft...",NaN,313000.0,0,1.5,Number of bathrooms: 1.5; Number of bedrooms: 3,"18810 Densmore Ave N, Shoreline, WA 98133, USA",20140502T000000,1955,3,0
1,"{'sqft_basement': 280, 'sqft_above': 3370, 'sq...",NaN,2384000.0,0,2.0,Number of bathrooms: 2.5; Number of bedrooms: 5,"709 W Blaine St, Seattle, WA 98119, USA",20140502T000000,1921,5,4
2,"{'sqft_basement': 0, 'sqft_above': 1930, 'sqft...",NaN,342000.0,0,1.0,Number of bathrooms: 2.0; Number of bedrooms: 3,"26206-26214 143rd Ave SE, Kent, WA 98042, USA",20140502T000000,1966,4,0
3,"{'sqft_basement': 1000, 'sqft_above': 1000, 's...",NaN,420000.0,0,1.0,Number of bathrooms: 2.25; Number of bedrooms: 3,"857 170th Pl NE, Bellevue, WA 98008, USA",20140502T000000,1963,4,0
4,"{'sqft_basement': 800, 'sqft_above': 1140, 'sq...",NaN,550000.0,0,1.0,Number of bedrooms: 4; Number of bathrooms: 2.5,"9105 170th Ave NE, Redmond, WA 98052, USA",20140502T000000,1976,4,0


#### - 📖 Column Reference

| Column | Meaning |
|---|---|
| `price` | Sale price of the house |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms (0.5 = half bathroom, no shower/tub) |
| `sqft_living` | Size of living area in square feet |
| `sqft_lot` | Size of the lot in square feet |
| `sqft_above` | Square footage of the house apart from the basement |
| `sqft_basement` | Square footage of the basement |
| `floors` | Number of floors |
| `waterfront` | 1 = has waterfront view, 0 = does not |
| `view` | 0–4 rating of how good the property's view is (0 = none, 4 = excellent) |
| `condition` | 1–5 rating of the house's overall condition (1 = poor, 5 = very good) |
| `yr_built` | Year the house was originally built |
| `yr_renovated` | Year of last renovation (NaN/0 = never renovated) |
| `date` | Date the house was sold |
| `address` | Property address |

#### - Looking at more rows
- 0.5 Bathroom means without bathtub/shower.

In [125]:
df["rooms"].head(10)      # looking at more rows


0     Number of bathrooms: 1.5; Number of bedrooms: 3
1     Number of bathrooms: 2.5; Number of bedrooms: 5
2     Number of bathrooms: 2.0; Number of bedrooms: 3
3    Number of bathrooms: 2.25; Number of bedrooms: 3
4     Number of bedrooms: 4; Number of bathrooms: 2.5
5     Number of bathrooms: 1.0; Number of bedrooms: 2
6     Number of bathrooms: 2.0; Number of bedrooms: 2
7     Number of bathrooms: 2.5; Number of bedrooms: 4
8     Number of bathrooms: 2.5; Number of bedrooms: 3
9     Number of bathrooms: 2.0; Number of bedrooms: 4
Name: rooms, dtype: str

#### - Looking inside one dictionary

In [126]:
df["area"].iloc[0]        

{'sqft_basement': 0,
 'sqft_above': 1340,
 'sqft_living/sqft_lot': 'sqft_living/sqft_lot=1340\\ 7912'}

#### - Flatten the nested `area` column

Extracts `sqft_basement` and `sqft_above` directly using `.str[...]` 
style dictionary access, and splits the combined string using pandas' 
built-in string methods.


In [127]:
df["sqft_basement"] = df["area"].str["sqft_basement"]
df["sqft_above"] = df["area"].str["sqft_above"]

living_lot = df["area"].str["sqft_living/sqft_lot"].str.split("=").str[1].str.split("\\", expand=True)
df["sqft_living"] = living_lot[0].str.strip().astype(int)
df["sqft_lot"] = living_lot[1].str.strip().astype(int)

df = df.drop(columns=["area"])
df.head()

,yr_renovated,price,waterfront,floors,rooms,address,date,yr_built,condition,view,sqft_basement,sqft_above,sqft_living,sqft_lot
0,NaN,313000.0,0,1.5,Number of bathrooms: 1.5; Number of bedrooms: 3,"18810 Densmore Ave N, Shoreline, WA 98133, USA",20140502T000000,1955,3,0,0,1340,1340,7912
1,NaN,2384000.0,0,2.0,Number of bathrooms: 2.5; Number of bedrooms: 5,"709 W Blaine St, Seattle, WA 98119, USA",20140502T000000,1921,5,4,280,3370,3650,9050
2,NaN,342000.0,0,1.0,Number of bathrooms: 2.0; Number of bedrooms: 3,"26206-26214 143rd Ave SE, Kent, WA 98042, USA",20140502T000000,1966,4,0,0,1930,1930,11947
3,NaN,420000.0,0,1.0,Number of bathrooms: 2.25; Number of bedrooms: 3,"857 170th Pl NE, Bellevue, WA 98008, USA",20140502T000000,1963,4,0,1000,1000,2000,8030
4,NaN,550000.0,0,1.0,Number of bedrooms: 4; Number of bathrooms: 2.5,"9105 170th Ave NE, Redmond, WA 98052, USA",20140502T000000,1976,4,0,800,1140,1940,10500


### 4. Parse the `rooms` text column
Splits the text on `;` to separate the two pieces of information.

In [128]:
# Split into two separate columns based on ";"
parts = df["rooms"].str.split(";", expand=True)
part0 = parts[0].str.strip()
part1 = parts[1].str.strip()

# Check which part contains "bathrooms" vs "bedrooms"
part0_is_bathrooms = part0.str.contains("bathrooms")

# Pick the correct piece for each, regardless of order
bathrooms_text = np.where(part0_is_bathrooms, part0, part1)
bedrooms_text = np.where(part0_is_bathrooms, part1, part0)

# Extract just the number after the colon
df["bathrooms"] = pd.Series(bathrooms_text).str.split(":").str[1].str.strip().astype(float)
df["bedrooms"] = pd.Series(bedrooms_text).str.split(":").str[1].str.strip().astype(int)

df = df.drop(columns=["rooms"])
df.head()

,yr_renovated,price,waterfront,floors,address,date,yr_built,condition,view,sqft_basement,sqft_above,sqft_living,sqft_lot,bathrooms,bedrooms
0,NaN,313000.0,0,1.5,"18810 Densmore Ave N, Shoreline, WA 98133, USA",20140502T000000,1955,3,0,0,1340,1340,7912,1.50,3
1,NaN,2384000.0,0,2.0,"709 W Blaine St, Seattle, WA 98119, USA",20140502T000000,1921,5,4,280,3370,3650,9050,2.50,5
2,NaN,342000.0,0,1.0,"26206-26214 143rd Ave SE, Kent, WA 98042, USA",20140502T000000,1966,4,0,0,1930,1930,11947,2.00,3
3,NaN,420000.0,0,1.0,"857 170th Pl NE, Bellevue, WA 98008, USA",20140502T000000,1963,4,0,1000,1000,2000,8030,2.25,3
4,NaN,550000.0,0,1.0,"9105 170th Ave NE, Redmond, WA 98052, USA",20140502T000000,1976,4,0,800,1140,1940,10500,2.50,4


#### - Handle `yr_renovated`

In [129]:
df["was_renovated"] = df["yr_renovated"].notna().astype(int)
df["yr_renovated"] = df["yr_renovated"].fillna(0)
df["yr_renovated"] = df["yr_renovated"].astype(int)

#### - Keeping only Date part from in date column

In [130]:
df["date"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d", errors="coerce")

#### Keeping only city in Address

In [131]:
df["city"] = df["address"].str.split(",").str[1].str.strip()
df = df.drop(columns=["address"])

#### - Check and handle missing values

In [132]:
missing = df.isnull().sum()
print(missing[missing > 0])
df = df.dropna(subset=["date"]) # Dropping the Nat dates (2 rows``)

date    2
dtype: int64


#### - Check and remove duplicates

In [133]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

Duplicates: 1


#### - Check data types

In [134]:
df.dtypes

yr_renovated              int64
price                   float64
waterfront                int64
floors                  float64
date             datetime64[us]
yr_built                  int64
condition                 int64
view                      int64
sqft_basement             int64
sqft_above                int64
sqft_living               int64
sqft_lot                  int64
bathrooms               float64
bedrooms                  int64
was_renovated             int64
city                     object
dtype: object

#### - Final Check

In [135]:
print("rows, columns : ",df.shape)
df = df[["city", "bedrooms", "bathrooms", "sqft_living", "sqft_lot",
         "sqft_above", "sqft_basement", "floors", "waterfront", "view",
         "condition", "yr_built", "yr_renovated", "was_renovated",
         "date", "price"]] # rearranging columns
df.head()

rows, columns :  (4598, 16)


,city,bedrooms,bathrooms,sqft_living,sqft_lot,sqft_above,sqft_basement,floors,waterfront,view,condition,yr_built,yr_renovated,was_renovated,date,price
0,Shoreline,3,1.50,1340,7912,1340,0,1.5,0,0,3,1955,0,0,2014-05-02,313000.0
1,Seattle,5,2.50,3650,9050,3370,280,2.0,0,4,5,1921,0,0,2014-05-02,2384000.0
2,Kent,3,2.00,1930,11947,1930,0,1.0,0,0,4,1966,0,0,2014-05-02,342000.0
3,Bellevue,3,2.25,2000,8030,1000,1000,1.0,0,0,4,1963,0,0,2014-05-02,420000.0
4,Redmond,4,2.50,1940,10500,1140,800,1.0,0,0,4,1976,0,0,2014-05-02,550000.0


#### - Statistical summary

In [136]:
print("Total Houses:", df.shape[0])
print("Average Price:", round(df["price"].mean(), 2))
print("Median Price:", round(df["price"].median(), 2))
print("Highest Price:", df["price"].max())
print("Lowest Price:", df["price"].min())
print("Average sqft_living:", round(df["sqft_living"].mean(), 2))
print("Average Bedrooms:", round(df["bedrooms"].mean(), 2))
print("Average Bathrooms:", round(df["bathrooms"].mean(), 2))
print("% Waterfront Properties:", round(df["waterfront"].mean() * 100, 2))
print("% Ever Renovated:", round(df["was_renovated"].mean() * 100, 2))

Total Houses: 4598
Average Price: 534565.97
Median Price: 453373.0
Highest Price: 26590000.0
Lowest Price: 0.0
Average sqft_living: 2139.46
Average Bedrooms: 3.4
Average Bathrooms: 2.16
% Waterfront Properties: 0.72
% Ever Renovated: 4.98


In [ ]:
# Saving the file (Uncomment the code below to save the File)

# df.to_csv("data/house_price_cleaned.csv", index=False)

---